In [ ]:
from langchain_postgres import PGVectorStore, PGEngine
from langchain_openai import OpenAIEmbeddings
import os

embeddings = OpenAIEmbeddings(
    model=os.environ['EMBEDDING_MODEL_NAME'],
    base_url=os.environ['BASE_URL'],
    api_key=os.environ['OPENAI_API_KEY'], # type: ignore
    check_embedding_ctx_length=False
)

engine = PGEngine.from_connection_string(url=os.environ['POSTGRES_URL'])

vector_store = PGVectorStore.create_sync(engine=engine, embedding_service=embeddings, table_name='langchain_test')
vector_store.similarity_search(query="客服", k=2)

In [ ]:
vector_store.as_retriever(search_kwargs=dict(k=2)).invoke(input="导购")

In [ ]:
from langchain.retrievers import MultiQueryRetriever
from langchain_openai import ChatOpenAI
import os
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

store_retriever = vector_store.as_retriever(search_kwargs=dict(k=1))

llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"], # type: ignore
)

retriever_from_llm = MultiQueryRetriever.from_llm(
    retriever=store_retriever,
    llm=llm,
)

retriever_from_llm.invoke('导购')

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_openai import ChatOpenAI
import os
from pydantic import SecretStr

llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=SecretStr(os.environ["OPENAI_API_KEY"]),
)
# 创建一个从 Document 中提取核心内容的 compressor
compressor = LLMChainExtractor.from_llm(llm)
# 创建一个会自动对上下文进行压缩的 Retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vector_store.as_retriever(search_kwargs=dict(k=2))
)
# vector_store.as_retriever(search_kwargs=dict(k=2)).invoke("安防")

compression_retriever.invoke("安防")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableSequence, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

SYSTEM_TEMPLATE='''
你是一个熟知内部知识库的机器人，你在回答时会引用知识库，并擅长通过自己的总结归纳，组织语言给出答案。
并且回答时仅根据知识库，尽可能回答用户问题，如果知识库中没有相关内容，你可以回答“原文中没有相关内容”，不要回答知识库以外的内容。

以下是知识库中跟用户回答相关的内容：
{context}

现在，你需要基于知识库，回答以下问题：
{question}
'''

prompt = ChatPromptTemplate.from_template(template=SYSTEM_TEMPLATE)

convert_docs_to_string = lambda docs: "".join(
    [f"{doc.page_content}\n" for doc in docs]
)

retriever_chain = compression_retriever | RunnableLambda(convert_docs_to_string)

def input_to_context(input):
    return {"question": input, "context": retriever_chain.invoke(input=input)}

rag_chain = RunnableSequence(
    first=RunnableLambda(input_to_context),
    middle=[
        prompt,
        llm,
    ],
    last=StrOutputParser(),
)

rst = rag_chain.invoke(input='客服')
print(rst)

In [ ]:
from langchain_core.prompts  import ChatPromptTemplate 
from langchain_core.output_parsers  import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.runnables import RunnableLambda
 
# 步骤1：定义回溯验证提示模板
validation_prompt = ChatPromptTemplate.from_messages([ 
    ("system", 
     """
     你是一个严谨的问答审核助手。请严格检查以下文档是否包含用户问题的答案：
     用户问题：{question}
     检索文档：{context}
     
     要求：
     1. 若文档明确包含问题答案，返回"YES"
     2. 若文档与问题无关或信息不足，返回"NO"
     """
    )
])
 
# 步骤2：定义安全生成提示模板
generation_prompt = ChatPromptTemplate.from_messages([ 
    ("system", 
     """
     根据审核结果和文档内容回答问题：
     用户问题：{question}
     文档审核结果：{validation_result}
     相关文档：{context}
     
     规则：
     - 若审核结果为"YES"，基于文档生成专业回答 
     - 若审核结果为"NO"，回复："根据现有资料无法回答该问题，建议补充相关资料。"
     - 禁止编造文档中不存在的信息
     """
    )
])

import os

from langchain_community.chat_models.tongyi import ChatTongyi

llm = ChatTongyi(
    model=os.environ["MODEL_NAME"],
    api_key=os.environ["OPENAI_API_KEY"], # type: ignore
    # streaming=True
)

embeddings = OpenAIEmbeddings(
    model=os.environ["EMBEDDING_MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"], # type: ignore
    check_embedding_ctx_length=False,
)
 
# 步骤3：构建回溯工作流
vector_store = FAISS.load_local(
    folder_path="../db/python/data",
    embeddings=embeddings,
    allow_dangerous_deserialization=True,
)

retriever = vector_store.as_retriever(search_kwargs={'k': 2})

convert_docs_to_string = lambda docs: "".join(
    [f"{doc.page_content}\n" for doc in docs]
)

retriever_chain = RunnableLambda(lambda x: x['question']) | retriever | RunnableLambda(convert_docs_to_string)

def validate_prompt_format(x):
    return {**x, "validation_prompt": validation_prompt.format_prompt(**x)}

def validation_invoke(x):
    return {"question": x['question'], "context": x['context'], "validation_result": (llm | StrOutputParser()).invoke(x['validation_prompt'])}

chain = (
    {"context": retriever_chain, "question": lambda x: x["question"]}
    | RunnableLambda(validate_prompt_format)
    | RunnableLambda(validation_invoke)
    | generation_prompt 
    | llm 
    | StrOutputParser()
)

# 步骤4：执行示例
response = chain.invoke({"question":  "你好"})

print(response)

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from pprint import pprint

embeddings = OpenAIEmbeddings(
    model=os.environ["EMBEDDING_MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"], # type: ignore
    check_embedding_ctx_length=False,
)

vector_store = FAISS.load_local(
    folder_path="../db/python/data",
    embeddings=embeddings,
    allow_dangerous_deserialization=True,
)

retriever = vector_store.as_retriever(search_kwargs={'k': 2})

rst = retriever.invoke('在线客服')

# rst = vector_store.similarity_search(query="在线客服", k=2)

pprint(rst)

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_core.output_parsers import StrOutputParser
import os

# 定义问题分解模板
template = """
【用户输入】：{question}
【当前已知信息】：{context}

【分解要求】：
1. 请将用户输入分解为2-3个递进的子问题，每个子问题需基于已知信息逐步深入
2. 每个子问题需明确“当前需要解决的具体问题”和“对最终答案的贡献”
3. 用自然语言描述子问题，格式为：“子问题x：[问题描述]（贡献：[说明改问题如何帮助回答原始问题]）”

【示例输出格式】：
子问题1： ...
子问题2： ...
"""

prompt_decomposition = ChatPromptTemplate.from_template(template)
# 调用大语言模型
llm = ChatTongyi(
    model=os.environ["MODEL_NAME"],
    api_key=os.environ["OPENAI_API_KEY"], # type: ignore
    # streaming=True
)
generate_queries_decomposition = (prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n")))
# 输入问题
question = "公司调岗，劳动合同主体变更可以拒绝吗?"
questions = generate_queries_decomposition.invoke({"question": question, "context": "劳动合同法"})
print(questions)

In [ ]:
from langchain_tavily import TavilySearch

tool = TavilySearch(max_results=2)
tools = [tool]

tool.invoke("What's a 'node' in LangGraph?")

#### 多模态RAG

In [ ]:
from unstructured.partition.pdf import partition_pdf
# from pdfminer.utils import open_filename

# 1. 加载PDF
elements = partition_pdf(filename="../documents/data.pdf")

print(elements)